In [ ]:
import pacmap 
import numpy        as np
import pandas       as pd
import umap.umap_   as umap

from tqdm                               import tqdm
from sklearn.preprocessing              import StandardScaler, MinMaxScaler
from sklearn.cluster                    import DBSCAN

from utils.Preprocessing                import PreProcessData
from utils.plotFigures                  import linePlot, lineAndScatterPlot, scatterPlot3D, scatterPlot2D
from clustering.plotClustering          import ClusteringResults

# Preprocessing

In [ ]:
preData = PreProcessData()
df1Raw  = preData.dataLoading(typeFile='h5')
preData.splitTimeSeries()
dfHealthy, dfUnhealthy = preData.getHealthyState()

scaler          = MinMaxScaler() 
scaledHealthy   = preData.normalizeData(data   = dfHealthy, 
                                        scaler = scaler)
scaledUnhealthy = preData.normalizeData(data   = dfUnhealthy,
                                        scaler = scaler)

In [ ]:
linePlot(data   = dfHealthy, 
         x      = 'tiempo', 
         y      = 'corriente(A)', 
         color  = 'trajectory',
         title  = 'Healthy Data')

linePlot(data   = dfUnhealthy, 
         x      = 'tiempo', 
         y      = 'corriente(A)', 
         color  = 'trajectory',
         title  = 'Unhealthy Data')

# Healthy Data Clustering

In [ ]:
dfManipulableVar = scaledHealthy[preData.manipulatedCr1]

clusteringModels    = {'umap':   DBSCAN(min_samples = 150,
                                        eps         = 2.2),

                       'pacmap': DBSCAN(min_samples = 200,
                                        eps         = 3  )} 

dimReductionModels  = {'umap':   umap.UMAP(n_components = 2, 
                                           random_state = 40),

                       'pacmap': pacmap.PaCMAP(n_components = 2, 
                                               random_state = 40, 
                                               save_tree    = True)}

In [ ]:
Clusters = ClusteringResults(data                = scaledHealthy, 
                             dataNormalized      = dfManipulableVar, 
                             clusteringModels    = clusteringModels, 
                             dimReductionModels  = dimReductionModels)

In [ ]:
# Get embedding space using Helathy Data
Clusters.train()

In [ ]:
# Get clusters for each trajectory
for trajectory in tqdm(Clusters.trajectories):
    Clusters.getResults(trajectory)

## Embedding Space

In [ ]:
Clusters.plotEmbedding(nfigures=2)

## Clusters Results

In [ ]:
Clusters.plotResults(nfigures=2)

# Unhealthy Data Clustering

In [ ]:
dfManipulableVar = scaledUnhealthy[preData.manipulatedCr1]

clusteringModels    = {'umap':   DBSCAN(min_samples = 150,
                                        eps         = 2.2),

                       'pacmap': DBSCAN(min_samples = 200,
                                        eps         = 3  )} 

dimReductionModels  = Clusters.dimReductionModels # Trained models

In [ ]:
ClustersUnhealthy = ClusteringResults(data                = scaledUnhealthy, 
                                      dataNormalized      = dfManipulableVar, 
                                      clusteringModels    = clusteringModels, 
                                      dimReductionModels  = dimReductionModels)

In [ ]:
ClustersUnhealthy.embeddingTrain = Clusters.embeddingTrain # trained embedding space with healthy data

In [ ]:
# Get clusters for each trajectory
for trajectory in tqdm(ClustersUnhealthy.trajectories):
    ClustersUnhealthy.getResults(trajectory)

In [ ]:
ClustersUnhealthy.plotEmbedding(nfigures=4)

In [ ]:
ClustersUnhealthy.plotResults(nfigures=4)